# Прокси-модель 2.2 — детекция начала аномального режима

**Трек M, гир 2.2 — только историческое исследование. Ни живого исполнения, ни заявок.**

Модель определяет **переход монеты из тихого (дефолтного) режима в аномальный** по трём метрикам на направленном исполнимом L1-спреде, всё **взвешено по времени** (вес тика = его dwell, длительность существования L1-состояния), без прореживания/сглаживания ряда.

Метрики (в **нормированной** `z+` и **абсолютной** `a+` переменной):
- амплитуда отклонения от верха тихого коридора `F75`: `z+ = max(0, s−F75)/σ`, `a+ = max(0, s−F75)`;
- occupancy `O_W` — доля времени окна `W` выше базового уровня;
- интегральная площадь `I_W` — накопленная амплитуда×длительность за окно `W`.

**Пороги** = высокие взвешенные по времени квантили каждой метрики по тихой референс-статистике за трейлинг `H_floor` (дефолт `Q_det=0.99`, один уровень на все монеты — численный порог самокалибруется per-coin). **Тишина — дефолт**, её не детектируем.

Всё настраивается в **первой ячейке**. Ниже — двухстадийный прогон: (1) чтение тиков + dwell-веса (медленно, кэшируется), (2) флор/метрики/пороги/детекция/plotly (быстро, повторяемо при перекрутке порогов).

Данные: реальная история — flat `spread_*.parquet` (см. `docs/model-data-sources.md`); `SYNTHETIC=True` генерит демо-день, если реальных данных ещё нет.

In [ ]:
# ============================================================================
#  ПАРАМЕТРЫ МОДЕЛИ 2.2 — единственная ячейка конфигурации
#  Меняй тут: пул монет, окно, пороги. Ниже ничего трогать не нужно.
# ============================================================================

# --- данные -----------------------------------------------------------------
SYNTHETIC   = True                      # True: сгенерить демо-день (нет реальных данных на этой VM)
DATA_ROOT   = "research_data/onset_sample_day"   # flat spread_*.parquet ИЛИ hive base_coin=/event_date=
DATA_LAYOUT = None                      # None = автодетект ("flat" / "hive")

# --- пул монет и временной интервал (UTC, END не входит) --------------------
COIN_POOL   = ["BTCX", "ETHX"]          # список тикеров рядом с параметрами
START       = "2026-08-21T00:00:00Z"
END         = "2026-08-22T00:00:00Z"
DIRECTIONS  = ["long", "short"]         # обе стороны считаем отдельно

# --- флор тихого коридора (взвешенные по времени квантили за H_floor) --------
H_FLOOR_H       = 12.0                  # трейлинг тихой истории, часы (демо ниже переопределяет на 3ч)
FLOOR_REFRESH_S = 60                    # каденция пересчёта медленного флора (НЕ сглаживание ряда)
Q_LO, Q_MID, Q_HI = 0.25, 0.50, 0.75    # коридор F25/F50/F75; σ = (F75−F25)/1.349
MIN_COVER_MIN   = 30                    # мин. взвешенное покрытие окна, чтобы флор был «тёплым»

# --- окно интегральных метрик -----------------------------------------------
W_MIN           = 30                    # окно occupancy O_W и площади I_W, минуты

# --- пороги = высокие квантили тихой статистики за H_floor -------------------
Q_DET   = 0.99                          # единый уровень; крути каждую метрику отдельно ниже
Q_AMP   = None                          # переопределение для амплитуды z_base/a_base (None → Q_DET)
Q_OCC   = None                          # переопределение для O_min
Q_AREA  = None                          # переопределение для I_min

# --- логика детекции ---------------------------------------------------------
COMBINE       = "or"                    # "norm_only" | "abs_only" | "and" | "or"
COOLDOWN_MIN  = 5                        # эпизод завершается после стольких минут без срабатывания
MERGE_GAP_MIN = 10                       # объединять эпизоды ближе этого зазора
WITH_VIZ_QUANTILES = True               # считать 90/95/99 квантили метрик для графиков

# --- качество данных / дыры -------------------------------------------------
MAX_GAP_MIN   = 5                        # разрыв длиннее → честная дыра (не тишина), сегмент рвётся
MAX_DWELL_S   = 10                       # клампа веса одного тика (пре-дырочный тик не доминирует)

# --- перф / кэш / вывод ------------------------------------------------------
WORKERS       = 4
STAGE1_CACHE  = "output/onset_cache"    # кэш тиков+dwell (стадия 1); '' — без кэша
VIZ_VARIABLE  = "norm"                   # какую переменную рисовать в графиках: "norm" | "abs"
SAVE_HTML_DIR = ""                       # напр. "output/onset_html" чтобы выгрузить графики; '' — не сохранять

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

# repo root on sys.path (run from repo root or research/)
REPO = Path.cwd()
if not (REPO / "research" / "anomaly_onset").exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from research.anomaly_onset import io_lean, detector as D, viz, synth

MS_H, MS_MIN, MS_S = 3600_000, 60_000, 1000

floor = D.FloorParams(
    H_floor_ms=int(H_FLOOR_H * MS_H), refresh_ms=int(FLOOR_REFRESH_S * MS_S),
    q_lo=Q_LO, q_mid=Q_MID, q_hi=Q_HI, min_cover_ms=int(MIN_COVER_MIN * MS_MIN),
)
metric = D.MetricParams(W_ms=int(W_MIN * MS_MIN))
thr = D.ThresholdParams(q_det=Q_DET, q_amp=Q_AMP, q_occ=Q_OCC, q_area=Q_AREA)
detect = D.DetectParams(
    combine=COMBINE, cooldown_ms=int(COOLDOWN_MIN * MS_MIN),
    merge_gap_ms=int(MERGE_GAP_MIN * MS_MIN), with_viz_quantiles=WITH_VIZ_QUANTILES,
)
dwell = io_lean.DwellParams(max_gap_ms=int(MAX_GAP_MIN * MS_MIN), max_dwell_ms=int(MAX_DWELL_S * MS_S))

data_root = REPO / DATA_ROOT
if SYNTHETIC:
    # demo day + a shorter floor so the whole warm region is stable (real default = H_FLOOR_H)
    data_root = REPO / "output" / "onset_synth_demo"
    synth.generate_day(data_root, synth.default_two_coins(), start=START,
                       hours=(io_lean.parse_ts_ms(END) - io_lean.parse_ts_ms(START)) / MS_H, seed=7)
    floor = D.FloorParams(H_floor_ms=3 * MS_H, refresh_ms=floor.refresh_ms,
                          q_lo=Q_LO, q_mid=Q_MID, q_hi=Q_HI, min_cover_ms=floor.min_cover_ms)
    print(f"[synthetic] demo day written to {data_root} | floor H=3h (real default {H_FLOOR_H}h)")

print("floor:", floor)
print("metric:", metric, "| thr:", thr, "| combine:", detect.combine)

## Стадия 1 — чтение тиков + dwell-веса (медленно, кэшируется)

Перечитывать сырые тики при перекрутке порогов не нужно: результат кэшируется. Меняешь только пороги в 1-й ячейке → перезапускаешь Стадию 2.

In [ ]:
def load_stage1(root, start, end, coins, dwell, workers, cache_dir):
    """Read lean L1 + derive spreads + add dwell weights (per-coin, hole-aware). Cached."""
    key = f"{Path(root).name}_{io_lean.parse_ts_ms(start)}_{io_lean.parse_ts_ms(end)}_{'-'.join(sorted(coins))}"
    cache = Path(cache_dir) / f"{key}.parquet" if cache_dir else None
    if cache and cache.exists():
        print(f"[stage1] cache hit {cache}")
        return pd.read_parquet(cache)
    df = io_lean.read_lean_ticks(root, start, end, coins=coins, workers=workers, layout=DATA_LAYOUT)
    df = io_lean.add_dwell_weights(df, dwell)
    if cache:
        cache.parent.mkdir(parents=True, exist_ok=True)
        df.to_parquet(cache)
        print(f"[stage1] cached -> {cache}")
    return df

ticks = load_stage1(data_root, START, END, COIN_POOL, dwell, WORKERS, STAGE1_CACHE)
print(f"[stage1] rows={len(ticks):,}  per-coin={ticks.groupby('base_coin').size().to_dict()}")
ticks[["event_dt", "base_coin", "spread_long", "spread_short", "dwell_ms", "segment"]].head()

## Стадия 2 — флор, метрики, пороги, детекция (быстро, повторяемо)

Прогон детектора по каждой монете × направлению. Результат: per-tick фреймы, каталог эпизодов, сводка.

In [ ]:
import time

frames, episodes, summary = {}, {}, []
for coin in COIN_POOL:
    g = ticks[ticks.base_coin == coin]
    if g.empty:
        print(f"[skip] {coin}: no ticks in window"); continue
    for direction in DIRECTIONS:
        t0 = time.perf_counter()
        fr = D.analyze(g, direction, floor=floor, metric=metric, thr=thr, detect=detect)
        ep = D.episodes_from_fire(fr, detect)
        dt = time.perf_counter() - t0
        frames[(coin, direction)] = fr
        episodes[(coin, direction)] = ep
        warm_from = fr.loc[fr.warm, "event_dt"].min() if fr.warm.any() else None
        summary.append({
            "coin": coin, "dir": direction, "ticks": len(fr),
            "warm_from": warm_from, "fire_ticks": int(fr.fire.sum()),
            "episodes": len(ep), "quiet_share": round(1 - fr.fire.mean(), 4),
            "I_norm_max": round(float(np.nanmax(fr.I_norm)) if len(fr) else np.nan, 4),
            "sec": round(dt, 2),
        })
summary = pd.DataFrame(summary)
print("detector runtime & counts:")
summary

In [ ]:
# Каталог эпизодов (все монеты/направления) + пороги, что дал квантиль (последнее значение)
cat = []
for (coin, direction), ep in episodes.items():
    e = ep.copy()
    if e.empty:
        continue
    e.insert(0, "dir", direction)
    e.insert(0, "coin", coin)
    cat.append(e)
episode_catalog = pd.concat(cat, ignore_index=True) if cat else pd.DataFrame()
print(f"episodes total: {0 if episode_catalog.empty else len(episode_catalog)}")
episode_catalog.head(30)

## Визуализация (plotly)

По каждой монице (переменная `VIZ_VARIABLE`):
1. **Тики + коридор** — все тики, взвешенная по времени скользящая средняя, коридор `Q25/Q50/Q75`, хвостовые квантили `90/95/99`.
2. **Амплитуда** `z+`/`a+` — с порогом `z_base` и квантилями `90/95/99`.
3. **Интегральные метрики** — `O_W` и `I_W` с порогами `O_min`/`I_min` и квантилями `90/95/99`.
4. **Overview перехода** — спред+коридор, амплитуда+порог, площадь+порог; подсветка найденных эпизодов.

In [ ]:
# выбери монету/направление для отрисовки
PLOT_COIN = COIN_POOL[0]
PLOT_DIR  = "long"

fr = frames[(PLOT_COIN, PLOT_DIR)]
ep = episodes[(PLOT_COIN, PLOT_DIR)]

save_dir = (REPO / SAVE_HTML_DIR) if SAVE_HTML_DIR else None
if save_dir:
    save_dir.mkdir(parents=True, exist_ok=True)

def _show(fig, tag):
    if save_dir:
        fig.write_html(save_dir / f"onset_{PLOT_COIN}_{PLOT_DIR}_{tag}.html")
    fig.show()

_show(viz.fig_ticks(fr, ep, coin=PLOT_COIN, direction=PLOT_DIR), "ticks")

In [ ]:
_show(viz.fig_amplitude(fr, ep, variable=VIZ_VARIABLE, coin=PLOT_COIN, direction=PLOT_DIR), "amplitude")

In [ ]:
_show(viz.fig_metrics(fr, ep, variable=VIZ_VARIABLE, coin=PLOT_COIN, direction=PLOT_DIR), "metrics")

In [ ]:
_show(viz.fig_overview(fr, ep, variable=VIZ_VARIABLE, coin=PLOT_COIN, direction=PLOT_DIR), "overview")